<a href="https://colab.research.google.com/github/TimofeyProtasov/diploma/blob/main/days/work_1304.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### dataset

In [2]:
from datasets import load_dataset

In [3]:
dataset = load_dataset('phatvo/hotpotqa-raft')

README.md:   0%|          | 0.00/651 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.30M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1855 [00:00<?, ? examples/s]

In [28]:
# Функция-фильтр: возвращает True, если oracle_context есть в instruction
def has_oracle_in_instruction(example):
    return example['oracle_context'] in example['instruction']

# Применяем фильтрацию
dataset_with_oracle = dataset['train'].filter(has_oracle_in_instruction)
dataset_without_oracle = dataset['train'].filter(lambda ex: not has_oracle_in_instruction(ex))

print(f"Примеров с золотым документом: {len(dataset_with_oracle)}")
print(f"Примеров без золотого документа: {len(dataset_without_oracle)}")

Filter:   0%|          | 0/1855 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1855 [00:00<?, ? examples/s]

Примеров с золотым документом: 1475
Примеров без золотого документа: 380


### model

In [34]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [35]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

In [36]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [37]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [42]:
def format_example(example):
    """
    Формирует текстовый промпт в формате чата Qwen2.5-Instruct.
    """
    # Создаём список сообщений, как того ожидает шаблон чата
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["cot_answer"]}
    ]
    # Применяем шаблон чата — он добавит все необходимые служебные токены
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,  # Пока не токенизируем, оставляем текстом
        add_generation_prompt=False  # Не добавляем маркер начала ответа, т.к. ответ уже есть
    )
    return {"formatted_text": formatted}

In [43]:
# Применяем функцию ко всему датасету
dataset_formatted = dataset_with_oracle.map(format_example)

# Посмотрим на первый отформатированный пример
print(dataset_formatted[0]["formatted_text"])

Map:   0%|          | 0/1475 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
<DOCUMENT>Glenmore Stratton Trenear-Harvey (born 29 December 1940) is a British intelligence analyst who writes, broadcasts and lectures on the subjects of security, intelligence, espionage and terrorism. He is the editor-in-chief of the "World Intelligence Review", an associate editor of "Eye Spy" intelligence magazine, and publisher of "Intelligence Digest". Trenear-Harvey is an intelligence analyst for Sky News, and also broadcasts on NBC, CNN, Al Jazeera, France 24, Russia Today, and the BBC. He hosted the weekly show "Energy World" several times, on the satellite channel Press TV. He claims to receive regular briefings from Britain’s Secret Intelligence Service (MI6) and Security Service (MI5) and maintains contact with former (and he claims serving) intelligence officers of the American, British, and former Soviet security and intelligence services.</DOCUMENT>
<DOCUM

In [44]:
dataset_formatted

Dataset({
    features: ['id', 'type', 'question', 'context', 'oracle_context', 'cot_answer', 'instruction', 'text', 'formatted_text'],
    num_rows: 1475
})

In [74]:
def tokenize_function(examples):
    """
    Токенизирует столбец 'formatted_text' и создаёт labels.
    """
    # Токенизируем текст, обрезая до максимальной длины контекста модели (32768 для Qwen2.5-0.5B)
    # Мы используем 1024 для экономии памяти и ускорения обучения
    tokenized = tokenizer(
        examples["formatted_text"],
        truncation=True,
        padding=True,           # Padding будем делать динамически в DataCollator
        max_length=1024,
        padding_side = 'right',
        truncation_side='left',  # Обрезаем токены слева
        return_tensors=None,     # Возвращаем обычные списки, не тензоры
    )
    # Для Causal LM метки совпадают с input_ids (модель предсказывает следующий токен)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

In [75]:
# Применяем токенизацию ко всему датасету
dataset_tokenized = dataset_formatted.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset_formatted.column_names,  # Удаляем старые текстовые столбцы для экономии памяти
)

print("Токенизация завершена. Пример размера:")
print(f"input_ids длина: {len(dataset_tokenized[0]['input_ids'])}")

Map:   0%|          | 0/1475 [00:00<?, ? examples/s]

Токенизация завершена. Пример размера:
input_ids длина: 1024


In [45]:
from peft import LoraConfig, get_peft_model, TaskType

In [46]:
# Настраиваем конфигурацию LoRA
lora_config = LoraConfig(
    r=8,                     # Ранг матриц адаптера (типичное значение — 8 или 16)
    lora_alpha=32,           # Коэффициент масштабирования (обычно вдвое больше r)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Слои, к которым применяется LoRA для Qwen2.5
    lora_dropout=0.1,       # Dropout для регуляризации адаптеров
    bias="none",             # Не добавляем обучаемое смещение
    task_type=TaskType.CAUSAL_LM,  # Тип задачи — языковое моделирование
)

In [48]:
!pip install -q triton

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.4 MB/s eta 0:00:00


In [51]:
# Применяем LoRA к модели
model = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [50]:
# Выведем информацию о количестве обучаемых параметров
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [54]:
from transformers import TrainingArguments

In [55]:
training_args = TrainingArguments(
    output_dir="./qwen-raft-lora",          # Папка для сохранения чекпоинтов
    num_train_epochs=3,                     # Количество проходов по датасету
    per_device_train_batch_size=2,          # Размер батча на одно устройство (GPU)
    gradient_accumulation_steps=2,          # Накапливаем градиенты 4 шага перед обновлением
    learning_rate=5e-5,                     # Скорость обучения (стандартное значение для LoRA)
    warmup_steps=5,                        # Шаги для постепенного увеличения LR в начале
    logging_steps=10,                       # Выводить лог каждые 10 шагов
    # save_steps=100,                         # Сохранять чекпоинт каждые 100 шагов
    save_total_limit=2,                     # Хранить только последние 2 чекпоинта
    fp16=True,                              # Использовать смешанную точность (быстрее и меньше памяти)
    report_to="none",                       # Отключаем внешние трекеры (wandb, tensorboard)
    remove_unused_columns=False,            # Не удалять лишние столбцы датасета
)

In [56]:
from transformers import Trainer, DataCollatorForLanguageModeling

In [57]:
# Создаём сборщик батчей для задачи языкового моделирования
# Он автоматически дополнит последовательности токенов до одинаковой длины внутри батча
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Мы НЕ используем Masked Language Modeling, у нас Causal LM
)

In [76]:
# Создаём тренер
trainer = Trainer(
    model=model,                     # Наша модель с LoRA
    args=training_args,              # Параметры обучения
    train_dataset=dataset_tokenized, # Токенизированный датасет
    data_collator=data_collator,     # Сборщик батчей
)

In [ ]:
# Запускаем обучение
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
